# CancerVerso · V3b — Más métricas de la U-Net (matriz de confusión, precisión y recall)
**No entrena nada.** Usa el modelo ya entrenado (`modelos/unet_t4/mejor.pt`) y los mismos 27,368 cortes de test (346 CTs, 193 pacientes que nunca vio).

Agrega las medidas "de libro" de machine learning que faltaban:
1. **Matriz de confusión por píxel** (fondo, hígado, tumor): de lo que marcó el experto, qué pintó el modelo.
2. **Precisión, recall e IoU** por clase: dicen *cómo* se equivoca (si pinta de más o si se le escapa).
3. **Matriz de confusión por corte:** ¿este corte tiene tumor, sí o no? Con sensibilidad y especificidad.
4. **La trampa de la exactitud:** cuánto sacaría un modelo que pinta todo como fondo.

**Entorno:** cualquier GPU (T4 basta). Tarda ~5 minutos. Solo hay que correr todo en orden (Entorno de ejecución → Ejecutar todas).
Al final guarda `metricas_extra.json` en `modelos/unet_t4/`.

## 1. Drive, GPU y configuración

In [ ]:
# ============================================================
# 1. MONTAR DRIVE, INSTALAR MONAI Y CONFIGURAR
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import subprocess
subprocess.run(["pip", "install", "-q", "monai"], check=True)

import os
import glob
import json
import shutil
import hashlib
import datetime

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from monai.networks.nets import UNet

if not torch.cuda.is_available():
    raise RuntimeError("No hay GPU. Entorno de ejecución -> Cambiar tipo de entorno -> T4 (o cualquier otra GPU).")
dispositivo = torch.device("cuda")
print("GPU:", torch.cuda.get_device_name(0))

BASE = "/content/drive/MyDrive/Big data Zucco"
DIR_DATASET = BASE + "/dataset_modelo/v2"       # shards del notebook V2
DIR_MODELO = BASE + "/modelos/unet_t4"          # el modelo que se evaluó en la V3 (corrida en T4)
LOCAL = "/content/cv_test"                      # copia local de los shards de test

# Misma precisión que en la V3: bf16 si la GPU lo soporta; si no (T4), fp16
if torch.cuda.get_device_capability()[0] >= 8:
    PRECISION = torch.bfloat16
else:
    PRECISION = torch.float16

CLASES = ["fondo", "hígado", "tumor"]
os.makedirs(LOCAL, exist_ok=True)

## 2. Copiar solo los shards de test y verificar sus huellas
Mismo control que en la V3: si la huella MD5 de un shard no coincide con el `manifest.json`, se detiene. Así sabemos que se evalúa exactamente el mismo test.

In [ ]:
# ============================================================
# 2. COPIA DE LOS SHARDS DE TEST + MD5
# ============================================================
with open(DIR_DATASET + "/manifest.json") as f:
    manifest = json.load(f)
archivos_test = [a for a in manifest["archivos"] if a["archivo"].startswith("test/")]
print(len(archivos_test), "shards de test")

def md5(ruta):
    h = hashlib.md5()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloque)
    return h.hexdigest()

malos = []
for a in archivos_test:
    origen = DIR_DATASET + "/" + a["archivo"]
    destino = LOCAL + "/" + a["archivo"]
    os.makedirs(os.path.dirname(destino), exist_ok=True)
    if not os.path.exists(destino):
        shutil.copy(origen, destino)
    if md5(destino) != a["md5"]:
        malos.append(a["archivo"])
if malos:
    raise RuntimeError("Huella MD5 distinta en: " + str(malos) + ". Borra esos archivos de " + LOCAL + " y vuelve a correr.")
print("Las", len(archivos_test), "huellas coinciden con el manifiesto.")

## 3. Cargar test y el modelo guardado

In [ ]:
# ============================================================
# 3. CARGAR TEST Y mejor.pt
# ============================================================
imagenes, etiquetas = [], []
for ruta in sorted(glob.glob(LOCAL + "/test/shard_*.npz")):
    s = np.load(ruta)
    imagenes.append(s["img"])
    etiquetas.append(s["lbl"])
img_test = np.concatenate(imagenes)
lbl_test = np.concatenate(etiquetas)
print("test:", img_test.shape)

# La misma U-Net de la V3
modelo = UNet(spatial_dims=2, in_channels=1, out_channels=3,
              channels=(32, 64, 128, 256, 512), strides=(2, 2, 2, 2), num_res_units=2).to(dispositivo)
modelo.load_state_dict(torch.load(DIR_MODELO + "/mejor.pt", map_location=dispositivo))
modelo.eval()
print("Modelo cargado de", DIR_MODELO + "/mejor.pt")

## 4. Predecir y contar
Por cada lote se cuentan dos cosas:
- **La matriz de confusión de píxeles:** una tabla de 3 × 3. La fila es lo que marcó el experto y la columna lo que pintó el modelo. Ejemplo: la casilla (tumor, hígado) cuenta los píxeles que eran tumor y el modelo pintó como hígado.
- **Por cada corte:** cuántos píxeles de tumor marcó el experto y cuántos pintó el modelo. Sirve para la matriz "¿este corte tiene tumor?".

In [ ]:
# ============================================================
# 4. PREDICCIÓN EN TEST
# ============================================================
matriz = torch.zeros(9, dtype=torch.int64, device=dispositivo)   # 3 x 3 aplanada
tumor_real_por_corte = []
tumor_pred_por_corte = []

with torch.no_grad():
    for i in range(0, len(img_test), 256):
        x = torch.from_numpy(img_test[i:i + 256]).to(dispositivo).unsqueeze(1).float() / 255
        y = torch.from_numpy(lbl_test[i:i + 256]).to(dispositivo).long()
        with torch.autocast("cuda", dtype=PRECISION):
            pred = modelo(x).argmax(dim=1)
        # Cada píxel cae en la casilla 3 * real + predicho (0 a 8)
        matriz += torch.bincount((3 * y + pred).flatten(), minlength=9)
        tumor_real_por_corte.append((y == 2).sum(dim=(1, 2)).cpu())
        tumor_pred_por_corte.append((pred == 2).sum(dim=(1, 2)).cpu())

matriz = matriz.reshape(3, 3).cpu().numpy()
tumor_real_por_corte = torch.cat(tumor_real_por_corte).numpy()
tumor_pred_por_corte = torch.cat(tumor_pred_por_corte).numpy()

tabla = pd.DataFrame(matriz, index=["experto: " + c for c in CLASES], columns=["modelo: " + c for c in CLASES])
print("Matriz de confusión (píxeles):")
print(tabla)

## 5. Precisión, recall, IoU y Dice por clase
Para cada clase (ejemplo: tumor):
- **VP** (verdaderos positivos): era tumor y el modelo pintó tumor.
- **FP** (falsos positivos): el modelo pintó tumor donde no había.
- **FN** (falsos negativos): era tumor y el modelo no lo pintó.

| Medida | Fórmula | En palabras |
|---|---|---|
| Precisión | VP ÷ (VP + FP) | De lo que pintó, cuánto era cierto |
| Recall (sensibilidad) | VP ÷ (VP + FN) | De lo que había, cuánto encontró |
| IoU (Jaccard) | VP ÷ (VP + FP + FN) | Intersección ÷ unión |
| Dice | 2·VP ÷ (2·VP + FP + FN) | Lo mismo que ya usamos; aquí sirve de control |

**Control:** el Dice de hígado y de tumor tiene que salir igual que en la V3 (0.949 y 0.653). Si no sale igual, algo cambió y no hay que usar estas cifras.

In [ ]:
# ============================================================
# 5. MEDIDAS POR CLASE
# ============================================================
por_clase = {}
for c, nombre in enumerate(CLASES):
    vp = int(matriz[c, c])
    fp = int(matriz[:, c].sum() - vp)      # toda la columna c menos la diagonal
    fn = int(matriz[c, :].sum() - vp)      # toda la fila c menos la diagonal
    por_clase[nombre] = {
        "precision": round(vp / max(vp + fp, 1), 4),
        "recall": round(vp / max(vp + fn, 1), 4),
        "iou": round(vp / max(vp + fp + fn, 1), 4),
        "dice": round(2 * vp / max(2 * vp + fp + fn, 1), 4),
        "pixeles_experto": int(matriz[c, :].sum()),
    }
print(pd.DataFrame(por_clase).T)

# Control contra la V3
dice_v3 = {"hígado": 0.949, "tumor": 0.6531}
for nombre, valor in dice_v3.items():
    diferencia = abs(por_clase[nombre]["dice"] - valor)
    print("Dice de", nombre, ":", por_clase[nombre]["dice"], "| V3:", valor, "| diferencia:", round(diferencia, 4))
    if diferencia > 0.005:
        raise RuntimeError("El Dice no coincide con la V3. Revisa que se cargó modelos/unet_t4/mejor.pt y el dataset v2.")
print("Coincide con la V3.")

# Matriz en porcentaje por fila: de lo que marcó el experto, qué % pintó el modelo como cada clase
matriz_pct = 100 * matriz / matriz.sum(axis=1, keepdims=True)

## 6. ¿Este corte tiene tumor? (matriz de confusión por corte)
Cada corte se vuelve una pregunta de sí o no:
- **Real:** el experto marcó al menos 1 píxel de tumor.
- **Modelo:** la U-Net pintó al menos 1 píxel de tumor.

Por qué por corte y no por CT: casi todos los CTs de test tienen tumor (340 de 346), así que una matriz por CT tendría solo 6 casos "sin tumor" y no diría nada. Por corte hay miles de cada lado.

In [ ]:
# ============================================================
# 6. MATRIZ POR CORTE: TUMOR SÍ / NO
# ============================================================
real_si = tumor_real_por_corte > 0
pred_si = tumor_pred_por_corte > 0

vp = int(np.sum(real_si & pred_si))       # tenía tumor y lo vio
fn = int(np.sum(real_si & ~pred_si))      # tenía tumor y no lo vio
fp = int(np.sum(~real_si & pred_si))      # no tenía y vio tumor
vn = int(np.sum(~real_si & ~pred_si))     # no tenía y no vio

por_corte = {
    "cortes": int(len(real_si)),
    "con_tumor": vp + fn,
    "sin_tumor": fp + vn,
    "vp": vp, "fn": fn, "fp": fp, "vn": vn,
    "sensibilidad": round(vp / max(vp + fn, 1), 4),     # de los cortes con tumor, cuántos detectó
    "especificidad": round(vn / max(vn + fp, 1), 4),    # de los cortes sin tumor, cuántos dejó en paz
    "precision": round(vp / max(vp + fp, 1), 4),        # cuando dice "hay tumor", cuántas veces es cierto
    "exactitud": round((vp + vn) / len(real_si), 4),
}
por_corte["f1"] = round(2 * por_corte["precision"] * por_corte["sensibilidad"]
                        / max(por_corte["precision"] + por_corte["sensibilidad"], 1e-9), 4)

print(pd.DataFrame([[vp, fn], [fp, vn]], index=["real: con tumor", "real: sin tumor"],
                   columns=["modelo: con tumor", "modelo: sin tumor"]))
for k in ["sensibilidad", "especificidad", "precision", "f1", "exactitud"]:
    print(k, ":", por_corte[k])

### 6b. ¿Qué tan grandes son las falsas alarmas?
Con la regla de "1 píxel basta", una manchita de pocos píxeles ya cuenta como "hay tumor". Aquí se mide el tamaño de esas falsas alarmas y qué pasa si se le exige al modelo un mínimo de píxeles para avisar: sube la especificidad (menos falsas alarmas) pero baja la sensibilidad (se le escapan más tumores). Es el mismo intercambio de cualquier prueba médica.

In [ ]:
# ============================================================
# 6b. TAMAÑO DE LAS FALSAS ALARMAS Y OTROS UMBRALES
# ============================================================
falsas = tumor_pred_por_corte[~real_si & pred_si]          # píxeles de tumor pintados en cortes sin tumor
falsas_alarmas = {
    "cortes": int(len(falsas)),
    "pixeles_mediana": float(np.median(falsas)),
    "pixeles_p75": float(np.percentile(falsas, 75)),
    "pixeles_p90": float(np.percentile(falsas, 90)),
}
print("Falsas alarmas:", falsas_alarmas)

umbrales_corte = []
for minimo in [1, 10, 50, 100, 200]:
    umbrales_corte.append({
        "min_pixeles": minimo,
        "especificidad": round(float(np.mean(tumor_pred_por_corte[~real_si] < minimo)), 4),
        "sensibilidad": round(float(np.mean(tumor_pred_por_corte[real_si] >= minimo)), 4),
    })
print(pd.DataFrame(umbrales_corte))

## 7. La trampa de la exactitud
La **exactitud** (aciertos ÷ total de píxeles) se ve muy alta en segmentación médica, pero engaña: el tumor es menos del 1% de los píxeles. Un "modelo" que pinta **todo como fondo** saca una exactitud altísima y no encuentra ni un tumor. Por eso el proyecto usa Dice, precisión y recall.

In [ ]:
# ============================================================
# 7. EXACTITUD DE LA U-NET VS UN MODELO QUE PINTA TODO DE FONDO
# ============================================================
total = int(matriz.sum())
exactitud_unet = np.trace(matriz) / total
exactitud_todo_fondo = matriz[0, :].sum() / total        # acierta solo los píxeles que de verdad eran fondo
pct_tumor = 100 * matriz[2, :].sum() / total

trampa = {
    "exactitud_unet": round(float(exactitud_unet), 4),
    "exactitud_todo_fondo": round(float(exactitud_todo_fondo), 4),
    "pct_pixeles_tumor": round(float(pct_tumor), 3),
}
print("Exactitud de la U-Net:", trampa["exactitud_unet"])
print("Exactitud de un modelo que pinta todo de fondo:", trampa["exactitud_todo_fondo"], "(y su recall de tumor es 0)")
print("% de píxeles que son tumor:", trampa["pct_pixeles_tumor"])

## 8. Gráficas y archivo de resultados

In [ ]:
# ============================================================
# 8. GRÁFICAS Y metricas_extra.json
# ============================================================
fig, (e1, e2) = plt.subplots(1, 2, figsize=(13, 5))
e1.imshow(matriz_pct, cmap="Blues", vmin=0, vmax=100)
for i in range(3):
    for j in range(3):
        e1.text(j, i, f"{matriz_pct[i, j]:.1f}%", ha="center", va="center",
                color="white" if matriz_pct[i, j] > 50 else "black")
e1.set_xticks(range(3), CLASES)
e1.set_yticks(range(3), CLASES)
e1.set_xlabel("Lo que pintó la U-Net")
e1.set_ylabel("Lo que marcó el experto")
e1.set_title("Píxeles (% por fila)")

m2 = np.array([[vp, fn], [fp, vn]])
e2.imshow(m2, cmap="Blues")
for i in range(2):
    for j in range(2):
        e2.text(j, i, f"{m2[i, j]:,}", ha="center", va="center",
                color="white" if m2[i, j] > m2.max() / 2 else "black")
e2.set_xticks(range(2), ["con tumor", "sin tumor"])
e2.set_yticks(range(2), ["con tumor", "sin tumor"])
e2.set_xlabel("U-Net")
e2.set_ylabel("Experto")
e2.set_title("Cortes: ¿tiene tumor?")
plt.tight_layout()
fig.savefig(DIR_MODELO + "/matrices_confusion.png", dpi=130)
plt.show()

resultado = {
    "_nota": "Métricas extra de la U-Net en test (notebook CancerVerso_V3b_metricas). Mismo modelo (mejor.pt) y mismo test que la V3. Cifras agregadas, sin IDs.",
    "gpu": torch.cuda.get_device_name(0),
    "matriz_pixeles": matriz.tolist(),
    "matriz_pixeles_pct_fila": np.round(matriz_pct, 2).tolist(),
    "por_clase": por_clase,
    "por_corte": por_corte,
    "falsas_alarmas": falsas_alarmas,
    "umbrales_corte": umbrales_corte,
    "trampa_exactitud": trampa,
    "creado": datetime.datetime.now().isoformat(timespec="seconds"),
}
with open(DIR_MODELO + "/metricas_extra.json", "w") as f:
    json.dump(resultado, f, indent=2, ensure_ascii=False)
print(json.dumps(resultado, indent=2, ensure_ascii=False))
print("\nGuardado en", DIR_MODELO + "/metricas_extra.json")